<a href="https://www.kaggle.com/code/unknowdont18181/deepfake-detection?scriptVersionId=317404239" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# ==========================================
# 0. INSTALL REQUIRED LIBRARIES (Runs once)
# ==========================================
import os
os.system("pip install gradio timm grad-cam opencv-python")

# ==========================================
# 1. IMPORTS & SETUP
# ==========================================
import gradio as gr
import torch
import torch.nn as nn
import timm
from torchvision import transforms
from PIL import Image
import numpy as np
import cv2

# Grad-CAM Imports
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

# Set device to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

# UPDATE THESE PATHS IF NEEDED
VIT_PATH = "/kaggle/input/datasets/unknowdont18181/deepfake-project/vit_hardfake_v2.pth"
EFF_PATH = "/kaggle/input/datasets/unknowdont18181/deepfake-project/eff_best.pth"

# ==========================================
# 2. LOAD MODELS
# ==========================================
print("Loading models... Please wait.")

# ViT Model
vit_model = timm.create_model('vit_base_patch16_224', pretrained=False)
vit_model.head = nn.Linear(vit_model.head.in_features, 2)
vit_model.load_state_dict(torch.load(VIT_PATH, map_location=device))
vit_model = vit_model.to(device).eval()

# EfficientNet Model
eff_model = timm.create_model('efficientnet_b4', pretrained=False)
eff_model.classifier = nn.Linear(eff_model.classifier.in_features, 2)
eff_model.load_state_dict(torch.load(EFF_PATH, map_location=device))
eff_model = eff_model.to(device).eval()

print("✅ Models successfully loaded!")

# ==========================================
# 3. GRAD-CAM & TRANSFORM SETUP
# ==========================================
target_layers = [eff_model.conv_head]
cam = GradCAM(model=eff_model, target_layers=target_layers)

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# ==========================================
# 4. FACIAL FEATURE DETECTION LOGIC
# ==========================================
def get_specific_issues(img_array, cam_map):
    issues = []
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_eye.xml')
    
    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)
    
    h, w = gray.shape
    cam_resized = cv2.resize(cam_map, (w, h))
    
    for (x, y, w_f, h_f) in faces:
        # Hair/Forehead
        hair_region = cam_resized[max(0, y-30):y+int(h_f*0.2), x:x+w_f]
        if hair_region.size > 0 and np.mean(hair_region) > 0.45:
            issues.append("• Hair/Forehead: Blurriness or blending artifacts near the hairline.")
            
        # Eyes
        roi_gray = gray[y:y+int(h_f*0.6), x:x+w_f]
        eyes = eye_cascade.detectMultiScale(roi_gray)
        for (ex, ey, ew, eh) in eyes:
            eye_cam = cam_resized[y+ey:y+ey+eh, x+ex:x+ex+ew]
            if eye_cam.size > 0 and np.mean(eye_cam) > 0.5:
                issues.append("• Eyes: Unnatural reflections or mismatched geometry detected.")
                break 
                
        # Mouth
        mouth_cam = cam_resized[y + int(h_f * 0.65):y + h_f, x + int(w_f * 0.25):x + int(w_f * 0.75)]
        if mouth_cam.size > 0 and np.mean(mouth_cam) > 0.5:
            issues.append("• Mouth/Teeth: Anomalies around the lips/teeth (common in lip-sync fakes).")
            
        # Jawline
        left_jaw = cam_resized[y:y+h_f, x:x+int(w_f*0.15)]
        right_jaw = cam_resized[y:y+h_f, x+int(w_f*0.85):x+w_f]
        if (left_jaw.size > 0 and np.mean(left_jaw) > 0.45) or (right_jaw.size > 0 and np.mean(right_jaw) > 0.45):
            issues.append("• Face Boundaries: Inconsistencies along the jawline (mask outlines).")
            
    # General / Background Fallback
    if not issues and np.mean(cam_map) > 0.4:
         issues.append("• General: Artificial textures detected across the broader image or background.")
            
    return "\n" + "\n".join(issues) if issues else "\n• No specific localized artifacts detected."

# ==========================================
# 5. CORE PREDICTION LOGIC (REAL-BIASED)
# ==========================================
def analyze_image(img_array):
    if img_array is None:
        return "ERROR", "Please upload an image.", None

    image = Image.fromarray(img_array).convert("RGB")
    img_tensor = transform(image).unsqueeze(0).to(device)

    # --- INFERENCE ---
    with torch.no_grad():
        vit_probs = torch.softmax(vit_model(img_tensor), dim=1)[0]
        eff_probs = torch.softmax(eff_model(img_tensor), dim=1)[0]

    # Balanced ensemble probabilities
    final_probs = 0.5 * eff_probs + 0.5 * vit_probs
    
    fake_prob = final_probs[0].item()
    real_prob = final_probs[1].item()

    # --- NEW LOGIC: BENEFIT OF THE DOUBT ---
    if fake_prob >= 0.65:
        label = "FAKE"
        confidence = fake_prob
    elif fake_prob >= 0.40 and fake_prob < 0.65:
        label = "REAL (MARGINAL)"
        confidence = real_prob
    else:
        label = "REAL"
        confidence = real_prob

    # --- GENERATE & RESIZE GRAD-CAM TO MATCH ORIGINAL IMAGE ---
    grayscale_cam = cam(input_tensor=img_tensor, targets=None)[0, :]
    original_h, original_w = img_array.shape[:2]
    grayscale_cam_resized = cv2.resize(grayscale_cam, (original_w, original_h))

    # --- TEXT REPORT GENERATION ---
    if label == "FAKE":
        specific_details = get_specific_issues(img_array, grayscale_cam_resized)
        desc = f"[!] AI MANIPULATION DETECTED (Confidence: {confidence*100:.1f}%)\n"
        desc += f"DETECTED ISSUES:{specific_details}"
        
    elif label == "REAL (MARGINAL)":
        desc = f"[-] BENEFIT OF THE DOUBT GIVEN (Real Confidence: {confidence*100:.1f}%)\n\n"
        desc += "Models detected minor localized anomalies, but lacked the high confidence required to flag this as a deepfake. Treated as broadly authentic."
        
    else:
        desc = f"[+] IMAGE APPEARS AUTHENTIC (Confidence: {confidence*100:.1f}%)\n\n"
        desc += "No significant artificial manipulation detected. Heatmap shows natural structural features."

    desc += f"\n\n--- DIAGNOSTICS ---\n> ViT Fake Prob: {vit_probs[0].item()*100:.1f}%\n> EffNet Fake Prob: {eff_probs[0].item()*100:.1f}%"

    # --- HEATMAP OVERLAY ---
    img_float = np.float32(img_array) / 255
    cam_image = show_cam_on_image(img_float, grayscale_cam_resized, use_rgb=True)

    return label, desc, cam_image

# ==========================================
# 6. CUSTOM UI DESIGN & LAUNCH
# ==========================================

custom_css = """
* {
    font-family: 'Courier New', Courier, monospace !important;
}
.verdict-text textarea {
    font-size: 32px !important;
    font-weight: bold !important;
    text-align: center !important;
    color: #ff4a4a !important; 
}
.header-text {
    text-align: center;
    border-bottom: 2px dashed #555;
    padding-bottom: 15px;
    margin-bottom: 25px;
}
.exec-btn {
    font-size: 18px !important;
    letter-spacing: 2px !important;
    border: 1px solid #555 !important;
}
"""

with gr.Blocks(theme=gr.themes.Monochrome(), css=custom_css) as demo:
    with gr.Column(elem_classes="header-text"):
        gr.Markdown("# 𝙳𝚎𝚎𝚙𝚏𝚊𝚔𝚎 𝙰𝚗𝚊𝚕𝚢𝚜𝚒𝚜 𝚃𝚎𝚛𝚖𝚒𝚗𝚊𝚕")
        gr.Markdown("𝙴𝚗𝚜𝚎𝚖𝚋𝚕𝚎 𝚅𝚒𝚜𝚒𝚘𝚗 𝚃𝚛𝚊𝚗𝚜𝚏𝚘𝚛𝚖𝚎𝚛 & 𝙴𝚏𝚏𝚒𝚌𝚒𝚎𝚗𝚝𝙽𝚎𝚝 𝙰𝚛𝚌𝚑𝚒𝚝𝚎𝚌𝚝𝚞𝚛𝚎")
    
    # TOP ROW: Fixed height images
    with gr.Row():
        with gr.Column():
            input_img = gr.Image(type="numpy", label="[ 𝙸𝙽𝙿𝚄𝚃_𝙸𝙼𝙰𝙶𝙴 ]", height=450)
        with gr.Column():
            output_cam = gr.Image(label="[ 𝙶𝚁𝙰𝙳-𝙲𝙰𝙼_𝙷𝙴𝙰𝚃𝙼𝙰𝙿 ]", interactive=False, height=450)
            
    # MIDDLE ROW: Full-width Execute Button
    with gr.Row():
        submit_btn = gr.Button(">> 𝙴𝚇𝙴𝙲𝚄𝚃𝙴_𝙰𝙽𝙰𝚕𝚈𝚂𝙸𝚂", variant="primary", size="lg", elem_classes="exec-btn")
            
    # BOTTOM ROW: Status and Diagnostics
    with gr.Row():
        with gr.Column(scale=1): 
            output_verdict = gr.Textbox(label="[ 𝚂𝚃𝙰𝚃𝚄𝚂 ]", elem_classes="verdict-text", lines=3)
        with gr.Column(scale=3): 
            output_desc = gr.Textbox(label="[ 𝙳𝙸𝙰𝙶𝙽𝙾𝚂𝚃𝙸𝙲_𝚁𝙴𝙿𝙾𝚁𝚃 ]", lines=8)

    submit_btn.click(
        fn=analyze_image,
        inputs=[input_img],
        outputs=[output_verdict, output_desc, output_cam]
    )

# Launch the app and generate the public URL
demo.launch(share=True)